# Entrenamiento simple y registro en MinIO

Objetivo: entrenar un modelo pequeño y guardar artefactos en un bucket de MinIO desde este notebook.

In [1]:
!pip install boto3 joblib

In [2]:
import json
from pathlib import Path

from joblib import dump
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
acc

0.9

In [1]:
import os
import boto3

artifact_dir = Path('artifacts')
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / 'iris_rf.joblib'
metrics_path = artifact_dir / 'metrics.json'

dump(model, model_path)
metrics_path.write_text(json.dumps({'accuracy': acc}, indent=2))

ModuleNotFoundError: No module named 'boto3'

In [4]:
endpoint = os.getenv('MINIO_ENDPOINT', 'http://minio:9000')
access_key = os.getenv('AWS_ACCESS_KEY_ID', 'admin')
secret_key = os.getenv('AWS_SECRET_ACCESS_KEY', 'supersecret')
bucket = 'mlops-minio-demo'

In [5]:
s3 = boto3.client(
    's3',
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
)

In [7]:
s3.upload_file(str(model_path), bucket, 'models/iris_rf.joblib')
s3.upload_file(str(metrics_path), bucket, 'metrics/metrics.json')

'Uploaded to MinIO'

'Uploaded to MinIO'

In [8]:
response = s3.list_objects_v2(Bucket=bucket)
[item['Key'] for item in response.get('Contents', [])]

['metrics/metrics.json', 'models/iris_rf.joblib']